# Movie Discovery Assistant - DPO Alignment Training

## Direct Preference Optimization (DPO)
DPO aligns the model with human preferences WITHOUT a reward model.

### Why DPO for Movie Recommendations?
- **No reward model needed** (simpler than PPO/RLHF)
- **Stable training** (no RL instability)
- **Works great with LoRA** (memory efficient)
- **Preference pairs** are easy to construct for recommendations

### Pipeline: SFT -> DPO
1. First: Train with SFT (supervised fine-tuning) - DONE
2. Then: Align with DPO using preference pairs

### Optimizations Applied:
- Flash Attention 2 (Ampere+ GPUs)
- 8-bit AdamW optimizer
- Gradient checkpointing (Unsloth)
- Cosine LR schedule
- BF16/FP16 auto-selection

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

if major_version >= 8:
    !pip install --no-deps packaging ninja einops "flash-attn>=2.5.0" xformers trl peft accelerate bitsandbytes
else:
    !pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

# ============================================================================
# Load SFT-trained model (starting point for DPO)
# ============================================================================
# DPO requires a reference model (the SFT model) and a policy model (to train)
# Unsloth handles this internally - it keeps ref model frozen

gpu_name = torch.cuda.get_device_name(0)
major_ver, _ = torch.cuda.get_device_capability()
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9

print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

dtype = torch.bfloat16 if major_ver >= 8 else torch.float16
max_seq_length = 2048

# Option 1: Load from HuggingFace (if you uploaded SFT model)
# model, tokenizer = FastLanguageModel.from_pretrained(
#     "your-username/movie-assistant-sft",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=True,
# )

# Option 2: Load base model + SFT LoRA adapters
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

print("Base model loaded. Apply SFT LoRA below.")

In [ ]:
# ============================================================================
# Apply LoRA for DPO (fresh adapters on top of SFT model)
# ============================================================================
# DPO LoRA can be smaller than SFT since we're fine-tuning preferences,
# not learning the task from scratch

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                              # Smaller rank for DPO (preference tuning)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

### Upload Preference Dataset

Upload `preference_train.jsonl` and `preference_val.jsonl` generated by `preference_data_generator.py`

Each line has: `{"prompt": ..., "chosen": ..., "rejected": ...}`

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload preference_train.jsonl and preference_val.jsonl

In [ ]:
from datasets import load_dataset

# Load preference data
train_dataset = load_dataset("json", data_files="preference_train.jsonl", split="train")
val_dataset = load_dataset("json", data_files="preference_val.jsonl", split="train")

print(f"Train preference pairs: {len(train_dataset)}")
print(f"Val preference pairs: {len(val_dataset)}")

# Preview a pair
sample = train_dataset[0]
print(f"\nSample prompt: {sample['prompt'][:100]}...")
print(f"Chosen (first 100): {sample['chosen'][:100]}...")
print(f"Rejected (first 100): {sample['rejected'][:100]}...")
print(f"Rejection strategy: {sample.get('rejection_strategy', 'unknown')}")

In [ ]:
from trl import DPOTrainer, DPOConfig

# ============================================================================
# DPO Training Configuration
# ============================================================================
# DPO loss: -log(sigmoid(beta * (log_pi(chosen) - log_pi(rejected))))
#
# beta controls how much the model can deviate from the reference:
#   - Higher beta (0.5) = Stay close to SFT model (conservative)
#   - Lower beta (0.05) = Allow more deviation (aggressive)
#   - 0.1 = Good default (balanced)
#
# For movie recommendations, beta=0.1 is ideal because:
#   - SFT model already knows the task format
#   - DPO just needs to refine preference ordering
#   - Too low = model becomes repetitive/degenerate
#   - Too high = no learning happens
# ============================================================================

use_bf16 = torch.cuda.is_bf16_supported()
batch_size = 4 if gpu_mem >= 15 else 2

dpo_config = DPOConfig(
    # --- DPO Specific ---
    beta=0.1,                          # KL penalty coefficient
    loss_type="sigmoid",               # Standard DPO loss
    max_prompt_length=512,
    max_length=1536,                   # prompt + completion

    # --- Training ---
    num_train_epochs=1,                # DPO needs fewer epochs
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,                # Lower LR for alignment
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,

    # --- Precision ---
    fp16=not use_bf16,
    bf16=use_bf16,

    # --- Logging ---
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,

    output_dir="dpo_outputs",
    report_to="none",
    seed=3407,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,                    # None = Unsloth uses implicit reference
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    args=dpo_config,
)

print("DPO Trainer configured:")
print(f"  Beta: {dpo_config.beta}")
print(f"  Loss: {dpo_config.loss_type}")
print(f"  LR: {dpo_config.learning_rate}")
print(f"  Batch: {batch_size} x 8 (grad accum)")

In [ ]:
# ============================================================================
# Train DPO
# ============================================================================
import time

start = time.time()
dpo_stats = dpo_trainer.train()
elapsed = time.time() - start

print(f"\n{'=' * 60}")
print(f"DPO TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"  Time: {elapsed/60:.1f} minutes")
print(f"  Loss: {dpo_stats.training_loss:.4f}")
print(f"  Steps: {dpo_stats.global_step}")
print(f"{'=' * 60}")

In [ ]:
# ============================================================================
# Evaluate: Compare SFT vs DPO outputs
# ============================================================================
FastLanguageModel.for_inference(model)

test_queries = [
    "Recommend a mind-bending sci-fi movie",
    "I want a comedy from the 90s",
    "Movies like The Matrix",
    "Dark thriller with a twist ending",
    "Something uplifting for a rainy day",
]

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

for query in test_queries:
    inputs = tokenizer(
        [alpaca_prompt.format(query, "", "")],
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
        )

    response = tokenizer.batch_decode(outputs)[0]
    clean = response.split("### Response:")[-1].replace(tokenizer.eos_token, "").strip()

    print(f"Q: {query}")
    print(f"A: {clean[:300]}")
    print("-" * 60)

In [ ]:
# ============================================================================
# Save DPO-aligned model
# ============================================================================
import shutil
from google.colab import files

# Save LoRA adapters
model.save_pretrained("dpo_lora_model")
tokenizer.save_pretrained("dpo_lora_model")

# Export GGUF for Ollama
print("Exporting to GGUF Q4_K_M...")
model.save_pretrained_gguf(
    "dpo_model_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

# Zip and download
shutil.make_archive('movie_assistant_dpo_lora', 'zip', 'dpo_lora_model')
shutil.make_archive('movie_assistant_dpo_gguf', 'zip', 'dpo_model_gguf')

files.download('movie_assistant_dpo_lora.zip')
files.download('movie_assistant_dpo_gguf.zip')
print("DPO model saved and exported!")